# Modulo 2 · Lezione 2 — Git locale avanzato

**Corso di Intelligenza Artificiale — Livello base**

---

## Cosa vedremo oggi?

| # | Blocco | Cosa imparerai |
|---|--------|----------------|
| 1 | `git diff` | Vedere esattamente cosa è cambiato e dove |
| 2 | Annullare i cambiamenti | `restore`, `revert`, `reset` — quando usare ciascuno |
| 3 | `git stash` | Salvare lavoro a metà senza commitare |
| 4 | `git commit --amend` | Correggere l'ultimo commit |
| 5 | `git log` avanzato | Filtrare e formattare lo storico |
| 6 | `git tag` | Contrassegnare versioni importanti |
| 7 | Pratica | Fare il refactoring di un progetto con uno storico pulito |

> 💡 **Idea centrale:** l'80% dei problemi che le persone hanno con Git non riguardano la collaborazione — riguardano «come annullo questo». Questa lezione copre esattamente questo, con la distinzione chiave tra operazioni **sicure** (che non riscrivono lo storico) e **pericolose** (che lo riscrivono).

In [ ]:
# Preparare l'ambiente di demo — creiamo un repo con storico esistente
import os, shutil
from pathlib import Path

REPO = Path.home() / "git_avanzato_demo"
if REPO.exists():
    shutil.rmtree(REPO)
REPO.mkdir()
os.chdir(REPO)

# Inizializzare e configurare
!git init -q
!git config user.name  "Studente Demo"
!git config user.email "demo@corso.com"

# Creare lo storico iniziale
Path(".gitignore").write_text("__pycache__/\n*.pyc\n.env\nvenv/\n")
Path("app.py").write_text("""
def saluta(nome):
    return f'Ciao, {nome}'

def congeda(nome):
    return f'Arrivederci, {nome}'
""")
Path("utils.py").write_text("""
def maiuscole(testo):
    return testo.upper()
""")

!git add -A && git commit -q -m "init: struttura iniziale"

# Secondo commit
with open("app.py", "a") as f:
    f.write("\ndef calcola(a, b):\n    return a + b\n")
!git add app.py && git commit -q -m "feat: aggiungere funzione calcola"

# Terzo commit
Path("config.py").write_text("DEBUG = True\nVERSION = '1.0'\n")
!git add config.py && git commit -q -m "chore: aggiungere configurazione"

print(f"✅ Repo demo pronto in {REPO}")
!git log --oneline

---
## 1. `git diff` — Vedere esattamente cosa è cambiato

`git diff` mostra le differenze tra diversi stati del repository. È il tuo strumento di revisione prima di commitare.

```
git diff                   →  working directory vs staging area
git diff --staged          →  staging area vs ultimo commit
git diff HEAD              →  working directory vs ultimo commit
git diff <hash1> <hash2>   →  tra due commit
git diff main..feature     →  tra due branch
```

**Leggere l'output di `git diff`:**
```diff
--- a/app.py        ← versione precedente
+++ b/app.py        ← versione nuova
@@ -3,4 +3,6 @@    ← righe 3-4 prima, 3-6 dopo
 def saluta(nome):         ← contesto (senza cambiamenti)
-    return f'Ciao'         ← riga eliminata (rosso)
+    return f'Ciao, {nome}'  ← riga aggiunta (verde)
```

In [ ]:
# Fare un cambiamento senza aggiungerlo allo stage
with open("app.py", "a") as f:
    f.write("\ndef nuova_funzione():\n    pass\n")

print("=== git diff (working vs stage) ===")
!git diff app.py

In [ ]:
!git add app.py

print("=== git diff --staged (stage vs ultimo commit) ===")
!git diff --staged app.py

In [ ]:
# Pulire lo stage per i prossimi esempi
!git restore --staged app.py
!git restore app.py

print("=== Diff tra primo e terzo commit ===")
!git diff HEAD~2 HEAD

In [ ]:
# Vedere solo il riepilogo (quali file e quante righe)
print("=== git diff --stat ===")
!git diff HEAD~2 HEAD --stat

print("\n=== git diff --name-only ===")
!git diff HEAD~2 HEAD --name-only

---
## 2. Annullare i cambiamenti — `restore`, `revert`, `reset`

Questa è la parte più importante della lezione. Ci sono tre comandi per annullare le cose e **non sono intercambiabili**.

```
COSA VUOI ANNULLARE?             COMANDO
─────────────────────────────────────────────────────────
Cambiamenti nel working          git restore <file>
directory
File aggiunto allo stage         git restore --staged <file>
Un commit già fatto (in modo     git revert <hash>        ← SICURO
sicuro, senza cancellare storico)
Tornare indietro spostando HEAD  git reset --soft  HEAD~n  ← annullare commit, mantenere stage
(con attenzione)                 git reset --mixed HEAD~n  ← annullare commit, mantenere working
                                  git reset --hard  HEAD~n  ← cancellare TUTTO ← DISTRUTTIVO
```

### La regola d'oro

> **Non usare mai `reset` su commit che hai già inviato a un repository remoto condiviso.**
> Usa sempre `revert` per annullare nello storico pubblico.

In [ ]:
# ── git restore: annullare cambiamenti nel working directory ─
print("PRIMA:")
print(Path("utils.py").read_text())

# Modificare il file
Path("utils.py").write_text("# FILE ROTTO\ndef maiuscole(testo):\n    return 'ERRORE'\n")
print("\nDOPO la modifica:")
print(Path("utils.py").read_text())

# Annullare — torna allo stato dell'ultimo commit
!git restore utils.py
print("DOPO git restore:")
print(Path("utils.py").read_text())

In [ ]:
# ── git restore --staged: togliere dallo stage senza perdere i cambiamenti ─
Path("utils.py").write_text("# Cambiamento importante\ndef maiuscole(testo):\n    return testo.upper()\n")
!git add utils.py

print("Stato dopo git add:")
!git status

# Togliere dallo stage — il cambiamento resta nel file
!git restore --staged utils.py
print("\nStato dopo git restore --staged:")
!git status
print("\nIl file ha ancora il cambiamento:")
print(Path("utils.py").read_text())

# Pulire per il prossimo esempio
!git restore utils.py

In [ ]:
# ── git revert: annullare un commit in modo SICURO ────────
# Crea un nuovo commit che inverte i cambiamenti del precedente
# Lo storico originale viene preservato — è SEMPRE sicuro

print("Storico prima del revert:")
!git log --oneline

# Revertire l'ultimo commit (quello di config.py)
!git revert HEAD --no-edit

print("\nStorico DOPO il revert:")
!git log --oneline
print("\nconfig.py non esiste più:")
!ls

In [ ]:
# ── git reset --soft: spostare HEAD senza toccare i file ───
# Ideale per: "voglio unire i miei ultimi N commit in uno solo"

# Prima ricreiamo il commit revertito per l'esempio
Path("config.py").write_text("DEBUG = True\nVERSION = '1.0'\n")
Path("extra.py").write_text("# Extra\n")
!git add config.py && git commit -q -m "chore: config"
!git add extra.py  && git commit -q -m "chore: extra"

print("Storico con due commit piccoli:")
!git log --oneline

# Annullare gli ultimi 2 commit ma mantenere i file nello stage
!git reset --soft HEAD~2

print("\nDopo reset --soft HEAD~2:")
!git log --oneline
print("\ngit status — i file restano nello stage:")
!git status

# Ora possiamo fare un solo commit pulito
!git commit -q -m "chore: aggiungere configurazione e modulo extra"
print("\nStorico pulito:")
!git log --oneline

In [ ]:
# ── Confronto dei tre tipi di reset ────────────────────────
confronto = """
╔══════════════╦══════════════╦══════════════╦══════════════╗
║ Comando      ║ Sposta HEAD  ║ Staging area ║ Working dir  ║
╠══════════════╬══════════════╬══════════════╬══════════════╣
║ --soft       ║ Sì           ║ Intatta      ║ Intatto      ║
║ --mixed      ║ Sì           ║ Pulita       ║ Intatto      ║
║ --hard       ║ Sì           ║ Pulita       ║ PULITA ⚠️    ║
╚══════════════╩══════════════╩══════════════╩══════════════╝

--soft  → annullare commit, mantenere TUTTO                (il più leggero)
--mixed → annullare commit e stage, mantenere i file       (il predefinito)
--hard  → annullare TUTTO, compresi i cambiamenti su disco (irreversibile)
"""
print(confronto)

---
## 3. `git stash` — Salvare lavoro a metà

Scenario classico: stai lavorando a una funzione a metà e all'improvviso devi cambiare branch per risolvere un bug urgente. Non vuoi commitare codice incompleto, ma nemmeno perderlo.

**`git stash`** salva i tuoi cambiamenti in una pila temporanea e lascia il working directory pulito.

```bash
git stash              # salvare i cambiamenti (push implicito)
git stash push -m "descrizione"   # con nome descrittivo
git stash list         # vedere tutti gli stash salvati
git stash pop          # recuperare il più recente ed eliminarlo
git stash apply        # recuperare senza eliminare dallo stash
git stash drop         # eliminare lo stash più recente
git stash clear        # eliminare tutti gli stash
```

In [ ]:
# Simulare lavoro a metà
with open("app.py", "a") as f:
    f.write("\n# TODO: questa funzione è a metà\ndef funzione_incompleta():\n    # ..."
            "\n    pass\n")
Path("bozza.py").write_text("# bozza temporanea\n")

print("Stato con lavoro a metà:")
!git status

In [ ]:
# Salvare nello stash con descrizione
!git add -A
!git stash push -m "WIP: funzione di esportazione a metà"

print("Stato dopo stash — working directory pulito:")
!git status

print("\nStash salvati:")
!git stash list

In [ ]:
# Simulare la risoluzione del bug urgente
with open("utils.py", "a") as f:
    f.write("\ndef minuscole(testo):\n    return testo.lower()\n")
!git add utils.py && git commit -q -m "fix: aggiungere funzione minuscole"

print("Bug risolto. Recuperiamo il lavoro dallo stash:")
!git stash pop

print("\nStato dopo pop — il lavoro torna:")
!git status

# Pulire
!git restore --staged . && git restore .
!git clean -fd

---
## 4. `git commit --amend` — Correggere l'ultimo commit

`--amend` sostituisce l'ultimo commit con uno nuovo. Usalo per:
- Correggere un errore di battitura nel messaggio.
- Aggiungere un file che avevi dimenticato di includere.
- Fare un piccolo cambiamento che dovrebbe far parte dello stesso commit.

> ⚠️ **Solo su commit che NON hai ancora inviato al remote.** `--amend` riscrive lo storico — se lo hai già inviato, causerà problemi ai tuoi colleghi.

In [ ]:
# Commit con messaggio errato
Path("nuovo_modulo.py").write_text("# Nuovo modulo\n")
!git add nuovo_modulo.py
!git commit -m "feat: nuov modulo"   # errore di battitura nel messaggio

print("Storico con messaggio errato:")
!git log --oneline

In [ ]:
# Correggere solo il messaggio
!git commit --amend -m "feat: aggiungere nuovo modulo"

print("Storico dopo amend:")
!git log --oneline

In [ ]:
# Aggiungere un file che avevamo dimenticato nel commit precedente
Path("nuovo_modulo_test.py").write_text("# Test del nuovo modulo\n")
!git add nuovo_modulo_test.py

# --no-edit mantiene il messaggio senza aprire l'editor
!git commit --amend --no-edit

print("Ora il commit include entrambi i file:")
!git show --stat HEAD

---
## 5. `git log` avanzato — Cercare nello storico

`git log` ha decine di opzioni. Queste sono le più utili nel lavoro quotidiano.

In [ ]:
# Creare altri commit per avere uno storico ricco
for i in range(3):
    Path(f"file_{i}.py").write_text(f"# File {i}\n")
    !git add . && git commit -q -m f"feat: file {i}"

print("=== Storico completo ===")
!git log --oneline

In [ ]:
# Filtrare per messaggio
print("=== Solo commit con 'feat' ===")
!git log --oneline --grep="feat"

print("\n=== Solo commit con 'chore' ===")
!git log --oneline --grep="chore"

In [ ]:
# -S cerca i commit che hanno aggiunto o rimosso una stringa di testo
# Utile per: in quale commit è stata introdotta questa funzione?
print("=== Commit che hanno toccato la stringa 'calcola' ===")
!git log --oneline -S "calcola"

In [ ]:
# Formato personalizzato
print("=== Formato personalizzato ===")
# %h=hash breve, %ad=data, %s=oggetto, %an=autore
!git log --pretty=format:"%h  %ad  %an  %s" --date=short

print("\n=== Con grafico dei branch ===")
!git log --oneline --graph --all

print("\n=== Ultimi 3 commit ===")
!git log --oneline -3

In [ ]:
# Vedere lo storico di un file specifico
print("=== Storico di app.py ===")
!git log --oneline -- app.py

print("\n=== Cosa è cambiato in app.py in ogni commit ===")
!git log -p -- app.py

In [ ]:
# git blame: chi ha scritto ogni riga e in quale commit
# Utile per capire perché qualcosa è scritto in un certo modo
print("=== git blame app.py ===")
!git blame app.py

---
## 6. `git tag` — Contrassegnare le versioni

I tag sono etichette permanenti che puntano a un commit specifico. Si usano per contrassegnare versioni del software: `v1.0`, `v2.3.1`, ecc.

| Tipo | Comando | Quando |
|------|---------|--------|
| **Lightweight** | `git tag v1.0` | Solo un'etichetta, senza metadati |
| **Annotated** | `git tag -a v1.0 -m "Versione 1.0"` | Con autore, data e messaggio — il consigliato |

In [ ]:
# Creare un tag annotato sul commit attuale
!git tag -a v1.0 -m "Prima versione stabile"

# Elencare i tag
print("=== Tag del repository ===")
!git tag

# Vedere i dettagli del tag
print("\n=== Dettagli del tag v1.0 ===")
!git show v1.0 --stat

# Nello storico compare il tag
print("\n=== Storico con tag ===")
!git log --oneline --decorate -5

In [ ]:
# Etichettare un commit precedente
HASH_SECONDO = !git log --oneline
hash_secondo = HASH_SECONDO[-2].split()[0]  # il secondo commit

!git tag -a v0.1 {hash_secondo} -m "Versione iniziale di prova"

print("Tag in ordine di versione:")
!git tag -l --sort=version:refname

print("\nStorico con tutti i tag:")
!git log --oneline --decorate

---
## 7. Pratica — Refactoring con storico pulito

L'obiettivo di questa pratica è applicare tutti i comandi della lezione in uno scenario realistico: ricevi un progetto con uno storico caotico e devi pulirlo e migliorarlo.

---

### Esercizio 1 — Esplorazione e diagnosi

**Consegna:** Usando il repo demo che hai già, rispondi a queste domande **solo con comandi Git** (senza aprire i file):

1. Quanti commit ha il repository?
2. In quale commit è stata introdotta la funzione `calcola`?
3. Quali sono i file attualmente tracciati?
4. Cosa è cambiato tra il primo e l'ultimo commit? (riepilogo dei file)
5. Chi è l'autore di ogni riga di `app.py`?

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 1</summary>

```python
# 1. Numero di commit
!git rev-list --count HEAD

# 2. Commit in cui è stata introdotta 'calcola'
!git log --oneline -S "calcola"

# 3. File tracciati
!git ls-files

# 4. Riepilogo dei cambiamenti tra primo e ultimo commit
!git diff HEAD~$(git rev-list --count HEAD | tr -d '\n')..HEAD --stat --diff-filter=A
# Forma più leggibile:
PRIMO = !git log --oneline | tail -1
hash_primo = PRIMO[0].split()[0]
!git diff {hash_primo} HEAD --name-status

# 5. Autore di ogni riga di app.py
!git blame app.py
```

**Nota:** `git log -S "calcola"` è il «pickaxe» — cerca i commit che hanno aggiunto o rimosso quella stringa esatta. Indispensabile per indagare quando è comparsa (o scomparsa) qualcosa nel codice.
</details>

---
### Esercizio 2 — Revertire senza perdere lo storico

**Consegna:**
1. Crea un file `errore.py` con contenuto errato e fai un commit: `"feat: modulo degli errori"`.
2. Verifica che compaia nello storico.
3. Usa `git revert` per annullare quel commit in modo sicuro.
4. Verifica che lo storico abbia ora **quattro commit** — l'originale più il revert — e che `errore.py` non esista più.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 2</summary>

```python
# 1. Creare il commit errato
Path("errore.py").write_text("# Questo modulo non dovrebbe esistere\n")
!git add errore.py
!git commit -m "feat: modulo degli errori"

# 2. Verificare nello storico
print("Storico con il commit errato:")
!git log --oneline

# 3. Revertire in modo sicuro
!git revert HEAD --no-edit

# 4. Verificare
print("\nStorico dopo revert (due commit nuovi, storico intatto):")
!git log --oneline
print("\nEsiste errore.py?")
!ls errore.py 2>/dev/null || echo "Non esiste — corretto"
```

**Perché `revert` e non `reset`:** `revert` aggiunge un nuovo commit che inverte i cambiamenti. Il commit errato resta nello storico — questo è positivo perché chiunque abbia clonato il repo vedrà esattamente cosa è successo. `reset --hard` cancellerebbe il commit dallo storico, il che causerebbe problemi se qualcuno lo ha già scaricato.
</details>

---
### Esercizio 3 — Pulire i commit con `reset --soft`

**Consegna:** Simula lo scenario di lavorare in modo disordinato e poi pulire:
1. Fai tre commit piccoli e mal nominati di fila:
   - `"wip"` → crea `modulo_a.py`
   - `"altri cambiamenti"` → crea `modulo_b.py`
   - `"quasi"` → crea `modulo_c.py`
2. Usa `git reset --soft HEAD~3` per unire i tre in uno solo.
3. Fai un commit pulito: `"feat: aggiungere moduli a, b e c"`.
4. Verifica lo storico.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 3</summary>

```python
# 1. Tre commit disordinati
Path("modulo_a.py").write_text("# A\n")
!git add modulo_a.py && git commit -q -m "wip"

Path("modulo_b.py").write_text("# B\n")
!git add modulo_b.py && git commit -q -m "altri cambiamenti"

Path("modulo_c.py").write_text("# C\n")
!git add modulo_c.py && git commit -q -m "quasi"

print("Storico disordinato:")
!git log --oneline

# 2. Unire i tre commit
!git reset --soft HEAD~3

print("\nDopo reset --soft — i file restano nello stage:")
!git status

# 3. Un commit pulito
!git commit -m "feat: aggiungere moduli a, b e c"

# 4. Storico pulito
print("\nStorico pulito:")
!git log --oneline
print("\nIl commit contiene i tre file:")
!git show --stat HEAD
```

**Questo pattern** — lavorare velocemente con commit WIP e poi pulire con `reset --soft` prima di inviare — è molto comune nel lavoro professionale. Nessuno nel team vede lo storico disordinato.
</details>

---
### Esercizio 4 — Il flusso completo con stash

**Consegna:** Simula lo scenario di interruzione più comune nel lavoro quotidiano:

1. Inizia a scrivere una nuova funzione in `app.py` (senza commitare).
2. «Ti arriva un bug urgente»: salva il tuo lavoro con `git stash push -m "WIP: funzione di esportazione"`.
3. Risolvi il bug: aggiungi una riga a `utils.py` e commita con `"fix: correggere funzione di pulizia"`.
4. Recupera il tuo lavoro dallo stash con `git stash pop`.
5. Termina la funzione e fai il commit finale.
6. Verifica lo storico completo.

In [ ]:
# ✏️ IL TUO CODICE QUI


<details>
<summary>💡 Vedi soluzione — Esercizio 4</summary>

```python
# 1. Lavoro a metà
with open("app.py", "a") as f:
    f.write("\n# TODO: terminare\ndef esporta_csv():\n    pass\n")
print("Stato con lavoro a metà:")
!git status

# 2. Salvare nello stash
!git add app.py
!git stash push -m "WIP: funzione di esportazione"
print("\nDopo stash — working directory pulito:")
!git status

# 3. Risolvere il bug urgente
with open("utils.py", "a") as f:
    f.write("\ndef rimuovi_spazi(testo):\n    return testo.strip()\n")
!git add utils.py
!git commit -m "fix: correggere funzione di pulizia"

# 4. Recuperare il lavoro dallo stash
!git stash pop
print("\nDopo stash pop — il lavoro torna:")
!git status

# 5. Terminare la funzione e commitare
contenuto = Path("app.py").read_text()
contenuto = contenuto.replace(
    "# TODO: terminare\ndef esporta_csv():\n    pass",
    "def esporta_csv(dati, percorso):\n    with open(percorso, 'w') as f:\n        f.write(','.join(dati))"
)
Path("app.py").write_text(contenuto)
!git add app.py
!git commit -m "feat: implementare esportazione in CSV"

# 6. Storico finale
print("\nStorico completo:")
!git log --oneline
```

**Osserva lo storico:** il commit del bug (`fix`) compare tra i due della nuova funzione (`stash` non lascia traccia nello storico). Il risultato è uno storico perfettamente ordinato anche se il lavoro reale è stato interrotto.
</details>

---
### Esercizio 5 (sfida) — Script di analisi dello storico

**Consegna:** Scrivi una funzione Python `analizza_storico(percorso_repo)` che usi `subprocess` per:

1. Rilevare commit con messaggi problematici: troppo corti (< 10 caratteri), senza prefisso `tipo:`, oppure che contengono parole come `"wip"`, `"temp"`, `"fix"`, `"asdf"`.
2. Rilevare commit molto grandi (> 5 file modificati).
3. Mostrare un report con suggerimenti: quali commit migliorare e come.

Non apporta alcuna modifica — analizza e riporta soltanto.

In [ ]:
# ✏️ IL TUO CODICE QUI
import subprocess


<details>
<summary>💡 Vedi soluzione — Esercizio 5</summary>

```python
import subprocess, re
from pathlib import Path

PREFISSI_VALIDI = {"feat", "fix", "docs", "refactor", "test", "chore", "init", "style", "perf"}
PAROLE_WIP      = {"wip", "temp", "tmp", "asdf", "xxx", "todo", "bozza"}

def git_cmd(args, cwd):
    r = subprocess.run(["git"] + args, cwd=cwd, capture_output=True, text=True)
    return r.stdout.strip()

def analizza_storico(percorso_repo):
    percorso = str(Path(percorso_repo).expanduser().resolve())

    # Ottenere tutti i commit: hash|messaggio
    log = git_cmd(["log", "--pretty=format:%h|%s"], percorso)
    if not log:
        print("Repository vuoto.")
        return

    problemi = []

    for riga in log.splitlines():
        hash_, _, messaggio = riga.partition("|")
        issues = []

        # Messaggio troppo corto
        if len(messaggio) < 10:
            issues.append(f"messaggio troppo corto ({len(messaggio)} caratteri)")

        # Senza prefisso convenzionale
        ha_prefisso = any(messaggio.lower().startswith(p + ":") for p in PREFISSI_VALIDI)
        if not ha_prefisso:
            issues.append("senza prefisso convenzionale (feat:, fix:, docs:…)")

        # Parole di lavoro in corso
        parole = set(re.findall(r'\w+', messaggio.lower()))
        wip_trovate = parole & PAROLE_WIP
        if wip_trovate:
            issues.append(f"contiene '{', '.join(wip_trovate)}'")

        # Commit con molti file
        stat = git_cmd(["show", "--stat", "--format=", hash_], percorso)
        file_cambiati = len([l for l in stat.splitlines() if "|") in l or " Bin" in l])
        if file_cambiati > 5:
            issues.append(f"troppi file ({file_cambiati}) — valuta se dividerlo")

        if issues:
            problemi.append((hash_, messaggio, issues))

    # Report
    totale = len(log.splitlines())
    print(f"\n{'═'*55}")
    print(f"  ANALISI DELLO STORICO — {Path(percorso).name}")
    print(f"  {totale} commit analizzati | {len(problemi)} con suggerimenti")
    print(f"{'═'*55}")

    if not problemi:
        print("  ✅ Storico impeccabile — nulla da migliorare")
    else:
        for hash_, messaggio, issues in problemi:
            print(f"\n  {hash_}  \"{messaggio}\"")
            for issue in issues:
                print(f"    ⚠️  {issue}")

    print(f"\n{'═'*55}")

analizza_storico("~/git_avanzado_demo")
```
</details>

---
## Riepilogo della lezione

```bash
# ─── Vedere i cambiamenti ──────────────────────────────────
git diff                        # working vs stage
git diff --staged               # stage vs ultimo commit
git diff <hash1> <hash2>        # tra due commit
git blame <file>                # chi ha scritto ogni riga

# ─── Annullare (dal meno al più distruttivo) ───────────────
git restore --staged <file>     # togliere dallo stage
git restore <file>              # scartare cambiamenti su disco
git revert <hash>               # annullare commit (SICURO)
git reset --soft  HEAD~n        # spostare HEAD, mantenere stage e disco
git reset --mixed HEAD~n        # spostare HEAD, mantenere solo disco
git reset --hard  HEAD~n        # cancellare TUTTO ⚠️

# ─── Stash ─────────────────────────────────────────────────
git stash push -m "descrizione" # salvare lavoro a metà
git stash list                  # vedere gli stash
git stash pop                   # recuperare ed eliminare
git stash apply                 # recuperare senza eliminare

# ─── Amend ─────────────────────────────────────────────────
git commit --amend -m "nuovo"   # correggere il messaggio
git commit --amend --no-edit    # aggiungere file dimenticato

# ─── Log avanzato ──────────────────────────────────────────
git log --oneline --graph --all
git log --grep="feat"           # filtrare per messaggio
git log -S "testo"              # commit che hanno toccato quella stringa
git log -p -- file.py           # storico di un file

# ─── Tag ───────────────────────────────────────────────────
git tag -a v1.0 -m "Versione 1.0"
git tag                         # elencare i tag
```

## Cosa vediamo nella prossima lezione?

Nella **Lezione 3** colleghiamo il repository locale al mondo: **GitHub e repository remoti** — `remote`, `clone`, `push`, `pull`, `fetch` e la differenza tra SSH e HTTPS.